# Day 4. Activation Function / ReLU / GELU

**오늘의 목표**
- Linear Layer만 쌓으면 왜 안 되는지 **직접 숫자로** 확인한다
- ReLU, GELU, Sigmoid, Tanh의 모양과 특징을 안다
- Activation이 들어가면 모델이 **곡선**을 표현할 수 있게 된다는 걸 실험으로 본다
- GPT의 FFN이 `Linear → GELU → Linear`인 이유를 설명할 수 있다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"     # 그래프에 한글 표시 (macOS)
plt.rcParams["axes.unicode_minus"] = False       # 마이너스 기호 깨짐 방지
torch.manual_seed(0)

## 1. 문제: Linear만 쌓으면 결국 Linear 하나

Linear Layer 두 개를 이어 붙이면:

```text
h = x W1          (첫 번째 Linear)
y = h W2          (두 번째 Linear)
  = (x W1) W2
  = x (W1 W2)     ← 행렬 곱은 묶는 순서를 바꿔도 결과가 같다
  = x W           ← W = W1 W2 인 Linear 하나와 똑같다!
```

10개, 100개를 쌓아도 마찬가지입니다. **층을 아무리 깊게 쌓아도 표현력이 늘지 않아요.**
직접 확인해봅시다. (설명을 단순하게 하려고 `bias=False`로 둡니다)

In [ ]:
layer1 = nn.Linear(3, 8, bias=False)
layer2 = nn.Linear(8, 2, bias=False)

x = torch.randn(5, 3)

# 두 층을 통과
two_layers = layer2(layer1(x))

# Weight 두 개를 미리 곱해서 '하나의 Weight'로 합치기
# (nn.Linear는 x @ weight.T 로 계산하니까 W1 = weight1.T, W2 = weight2.T)
W_merged = layer1.weight.T @ layer2.weight.T      # (3, 8) @ (8, 2) → (3, 2)
one_layer = x @ W_merged

print("합친 Weight shape:", tuple(W_merged.shape))
print("두 층 결과 == 한 층 결과 ?", torch.allclose(two_layers, one_layer, atol=1e-6))

**해결책**: Linear 사이에 **비선형 함수(Activation Function)** 를 끼워 넣습니다.

```text
Linear → Activation → Linear
```

이러면 더 이상 하나로 합칠 수 없게 됩니다.

In [ ]:
with_act = layer2(F.relu(layer1(x)))
print("Activation 넣은 결과 == 한 층 결과 ?", torch.allclose(with_act, one_layer, atol=1e-6))

## 2. 대표적인 Activation Function 4가지

전부 **숫자 하나를 받아서 숫자 하나를 돌려주는 함수**입니다.
텐서에 적용하면 **원소 하나하나에 따로** 적용됩니다. 그래서 shape는 절대 바뀌지 않아요.

| 이름 | 식 | 출력 범위 | 주로 쓰는 곳 |
|---|---|---|---|
| ReLU | `max(0, x)` | `0 ~ ∞` | CNN, 일반 신경망의 기본값 |
| GELU | `x · Φ(x)` | 약 `-0.17 ~ ∞` | **GPT, BERT 등 Transformer** |
| Sigmoid | `1 / (1 + e⁻ˣ)` | `0 ~ 1` | 확률 하나, 게이트 |
| Tanh | `(eˣ - e⁻ˣ) / (eˣ + e⁻ˣ)` | `-1 ~ 1` | RNN, GELU 근사식 |

(`Φ(x)`는 표준정규분포에서 x보다 작은 값이 나올 확률입니다. 지금은 몰라도 괜찮아요.)

### 2-1. 값으로 먼저 보기

In [ ]:
x = torch.tensor([-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0])

print("x       :", x.tolist())
print("ReLU    :", [round(v, 3) for v in F.relu(x).tolist()])
print("GELU    :", [round(v, 3) for v in F.gelu(x).tolist()])
print("Sigmoid :", [round(v, 3) for v in torch.sigmoid(x).tolist()])
print("Tanh    :", [round(v, 3) for v in torch.tanh(x).tolist()])

**관찰 포인트**
- **ReLU**: 음수는 전부 0, 양수는 그대로
- **GELU**: ReLU와 비슷하지만 0 근처가 **부드럽고**, 작은 음수는 살짝 음수로 남김 (`-0.5 → -0.154`)
- **Sigmoid**: 아무리 큰 수가 들어와도 0~1 사이로 눌러버림
- **Tanh**: -1~1 사이로 눌러버림, 0을 중심으로 대칭

### 2-2. 그래프로 보기

In [ ]:
x = torch.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, fn) in zip(axes, [
    ("ReLU", F.relu), ("GELU", F.gelu), ("Sigmoid", torch.sigmoid), ("Tanh", torch.tanh)
]):
    ax.plot(x, fn(x), linewidth=2.5)
    ax.plot(x, x, "--", color="gray", alpha=0.4, label="y = x (Linear)")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(name, fontsize=14)
    ax.set_ylim(-2, 4)
    ax.grid(alpha=0.3)
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

회색 점선 `y = x`는 **직선**입니다. 네 함수 모두 어딘가에서 **꺾이거나 휘어** 있죠?
이 꺾임 덕분에 여러 층을 쌓았을 때 하나로 합쳐지지 않습니다.

### 2-3. ReLU vs GELU 자세히 비교

GPT가 ReLU 대신 GELU를 쓰는 이유는 0 근처에서 보입니다.

In [ ]:
x = torch.linspace(-3, 2, 200)
plt.figure(figsize=(7, 4))
plt.plot(x, F.relu(x), linewidth=2.5, label="ReLU")
plt.plot(x, F.gelu(x), linewidth=2.5, label="GELU")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.title("ReLU vs GELU", fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

| | ReLU | GELU |
|---|---|---|
| 0에서 | **뾰족하게 꺾임** | **부드럽게 휨** |
| 음수 입력 | 무조건 0 (정보가 완전히 사라짐) | 작은 음수는 조금 남김 |
| 계산 | 매우 쉬움 | 조금 더 복잡 |

부드러운 곡선은 학습(Day 7의 Gradient 계산)을 더 안정적으로 만들어줍니다.
실험적으로 Transformer에서 GELU가 조금 더 좋은 성능을 내서 GPT-2부터 표준처럼 쓰이고 있어요.

## 3. 실험: Activation이 있어야 곡선을 배울 수 있다

`y = sin(x)` 곡선을 두 모델로 학습시켜 봅시다.
- **모델 A**: Linear → Linear → Linear (Activation 없음)
- **모델 B**: Linear → **GELU** → Linear → **GELU** → Linear

학습 루프는 Day 7에서 자세히 배우니, 지금은 결과 그래프만 보세요.

In [ ]:
X = torch.linspace(-3, 3, 100).unsqueeze(1)   # (100, 1)
Y = torch.sin(X)

def make_model(use_activation):
    act = nn.GELU() if use_activation else nn.Identity()   # Identity = 아무것도 안 함
    return nn.Sequential(
        nn.Linear(1, 32), act,
        nn.Linear(32, 32), act,
        nn.Linear(32, 1),
    )

def train(model, steps=2000):
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(steps):
        loss = ((model(X) - Y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()

model_A = make_model(use_activation=False)
model_B = make_model(use_activation=True)
loss_A = train(model_A)
loss_B = train(model_B)
print(f"모델 A (Activation 없음) 최종 Loss: {loss_A:.5f}")
print(f"모델 B (GELU 있음)       최종 Loss: {loss_B:.5f}")

In [ ]:
with torch.no_grad():
    plt.figure(figsize=(8, 4))
    plt.plot(X, Y, "k--", linewidth=2, label="정답: sin(x)")
    plt.plot(X, model_A(X), linewidth=2.5, label=f"모델 A: Linear만 (loss {loss_A:.3f})")
    plt.plot(X, model_B(X), linewidth=2.5, label=f"모델 B: GELU 포함 (loss {loss_B:.5f})")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title("Activation이 있어야 곡선을 표현할 수 있다", fontsize=13)
    plt.show()

모델 A는 층이 3개인데도 **직선 하나**밖에 못 그립니다. (1절에서 본 대로 Linear 하나랑 똑같으니까요)
모델 B는 GELU 덕분에 sin 곡선을 거의 그대로 따라갑니다.

> 💡 **핵심**: Activation Function = 신경망이 **직선이 아닌 복잡한 관계**를 배울 수 있게 해주는 장치

## 4. LLM과의 연결: FFN (Feed Forward Network)

GPT의 Transformer Block 안에는 이런 작은 신경망이 들어있습니다. (커리큘럼 6-7)

```python
nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),   # 벡터 길이를 4배로 늘리고
    nn.GELU(),                       # 비선형성 추가
    nn.Linear(4 * n_embd, n_embd)    # 다시 원래 길이로
)
```

Day 3 연습 문제 Q3에서 만든 `up`(32→128) / `down`(128→32) 사이에 GELU를 끼운 것이에요.

In [ ]:
B, T, n_embd = 2, 4, 32

ffn = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.GELU(),
    nn.Linear(4 * n_embd, n_embd),
)

x = torch.randn(B, T, n_embd)
h1 = ffn[0](x)
h2 = ffn[1](h1)
out = ffn[2](h2)

print("입력          :", tuple(x.shape))
print("Linear (×4)   :", tuple(h1.shape))
print("GELU          :", tuple(h2.shape), " ← Activation은 shape를 안 바꿈")
print("Linear (÷4)   :", tuple(out.shape))
print("파라미터 개수 :", sum(p.numel() for p in ffn.parameters()))

> 💡 GELU 없이 Linear 두 개만 있었다면, 1절에서 본 대로 32→32짜리 Linear 하나와 같아집니다.
> 4배로 늘렸다 줄이는 게 아무 의미가 없어지는 거죠. **GELU가 있어서 FFN이 의미를 갖습니다.**

## 5. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 아래 입력에 ReLU를 적용한 결과를 **손으로** 적으세요. (실행 전에 예측)

In [ ]:
x = torch.tensor([-2.0, 0.0, 3.0, -0.1, 1.5])

my_answer = "?"   # TODO: 리스트로. 예) [1.0, 2.0, ...]

assert my_answer == F.relu(x).tolist(), "ReLU의 정의를 다시 떠올려보세요"
print("Q1 정답!")

**Q2.** `F.relu`를 쓰지 않고 ReLU를 직접 구현하세요.
힌트: `torch.maximum(a, b)` 또는 `torch.clamp(x, min=...)` 또는 `x * (x > 0)`

In [ ]:
def my_relu(x):
    return None  # TODO

x = torch.randn(3, 4)
assert torch.equal(my_relu(x), F.relu(x))
print("Q2 정답!")

**Q3.** GELU를 직접 구현하세요. 정확한 식은 아래와 같습니다.

```text
GELU(x) = 0.5 · x · (1 + erf(x / √2))
```

`erf`는 `torch.erf()`로, `√2`는 `2 ** 0.5`로 계산할 수 있어요.

In [ ]:
def my_gelu(x):
    return None  # TODO

x = torch.linspace(-4, 4, 50)
assert torch.allclose(my_gelu(x), F.gelu(x), atol=1e-6)
print("Q3 정답!")

**Q4.** Activation 없이 Linear 3개를 쌓은 모델을 **Linear 하나로 합치세요.**
`merged`는 `nn.Linear(4, 2, bias=False)`이고, `merged.weight`에 알맞은 값을 넣으면 됩니다.

힌트: 1절 코드를 참고하세요. `nn.Linear`는 `x @ weight.T`로 계산하고, `merged.weight`의 shape는 `(2, 4)`여야 해요.

In [ ]:
l1 = nn.Linear(4, 6, bias=False)
l2 = nn.Linear(6, 5, bias=False)
l3 = nn.Linear(5, 2, bias=False)
deep = nn.Sequential(l1, l2, l3)

merged = nn.Linear(4, 2, bias=False)
with torch.no_grad():
    merged.weight.copy_(None)   # TODO: None 자리에 합친 weight

x = torch.randn(10, 4)
assert torch.allclose(deep(x), merged(x), atol=1e-5)
print("Q4 정답! 층 3개짜리 모델 = Linear 하나")

## 6. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. 왜 Linear Layer만 계속 쌓으면 안 되는가?
2. ReLU는 어떤 함수인가? 음수와 양수 입력을 각각 어떻게 처리하는가?
3. GELU는 ReLU와 무엇이 다른가? GPT는 왜 GELU를 쓰는가?
4. Activation Function을 텐서에 적용하면 shape는 어떻게 되는가?
5. GPT의 FFN에서 GELU를 빼면 어떤 일이 생기는가?

**내 답:**

1.
2.
3.
4.
5.